# 复现回测结果

验证本地回测与 `--from-predictions` 结果一致。

In [2]:
import os, sys, pickle, json
import pandas as pd
import numpy as np
sys.path.insert(0, '../code/src')
from backtest import ETFBacktester, run_backtest, run_backtest_from_predictions
import warnings
warnings.filterwarnings('ignore')

## 配置

In [3]:
MODEL_DIR = "../model/search_itransformer_74_3/exp_54"
MODEL_FILE = "best_model_sliding.pth"
DATA_PATH = "../etf_data/etf_74.csv"
CACHE_DIR = "../output/predictions_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

START_DATE = "2026-04-01"
END_DATE = "2026-05-12"
TOP_K = 3
REBALANCE_DAYS = 5
POSITION_PCT = 0.95
INITIAL_CAPITAL = 100000

## 方法 A: 直接加载模型回测（传统方式）

In [4]:
for mode in ['close', 'open']:
    result = run_backtest(
        model_dir=MODEL_DIR, data_path=DATA_PATH,
        start_date=START_DATE, end_date=END_DATE,
        top_k=TOP_K, rebalance_days=REBALANCE_DAYS,
        position_pct=POSITION_PCT, initial_capital=INITIAL_CAPITAL,
        trade_mode=mode, model_file=MODEL_FILE,
        verbose=False, log=False, device='cpu',
    )
    print(f'[A] {mode:5s}: return={result.strategy_return:.2f}%  dd={result.max_drawdown:.2f}%  hs300={result.hs300_return:.2f}%  win={result.rebalance_stats['win_rate']:.1f}%  avg={result.rebalance_stats['avg_return']:+.2f}%')

[A] close: return=27.31%  dd=3.04%  hs300=9.53%  win=100.0%  avg=+3.87%
[A] open : return=11.44%  dd=11.03%  hs300=9.53%  win=25.0%  avg=+0.53%


## 方法 B: 先生成预测缓存，再快速回测

### Phase 1: 生成预测信号（加载模型，慢）

In [5]:
import time
t0 = time.time()

cached_data, cached_features = ETFBacktester.load_data_once(
    data_path=DATA_PATH,
    scaler_path=f'{MODEL_DIR}/scaler.pkl',
    feature_num='39',
    verbose=True,
)

bt = ETFBacktester.from_cached_data(
    model_dir=MODEL_DIR, cached_data=cached_data,
    cached_features=cached_features, device='cpu',
    model_file=MODEL_FILE, verbose=False,
)

preds_dict = bt.generate_predictions_dict(start_date=START_DATE, end_date=END_DATE, rebalance_days=REBALANCE_DAYS, first_rebalance_date=START_DATE)
print(f'生成 {len(preds_dict)} 个交易日的预测信号')

# 缓存
cache_path = f'{CACHE_DIR}/search_itransformer_74_3_exp_54_{MODEL_FILE}.pkl'
with open(cache_path, 'wb') as f:
    pickle.dump(preds_dict, f)

del bt.model, bt
print(f'模型已释放, 缓存已保存: {cache_path}')
print(f'Phase 1 耗时: {time.time()-t0:.1f}s')

使用缓存数据: ../etf_data/etf_74.csv
生成 9 个交易日的预测信号
模型已释放, 缓存已保存: ../output/predictions_cache/search_itransformer_74_3_exp_54_best_model_sliding.pth.pkl
Phase 1 耗时: 2.5s


### Phase 2: 从缓存回测（秒级）

In [6]:
import time

with open(cache_path, 'rb') as f:
    preds_dict = pickle.load(f)

for mode in ['close', 'open']:
    t0 = time.time()
    result = run_backtest_from_predictions(
        predictions_dict=preds_dict, data_path=DATA_PATH,
        start_date=START_DATE, end_date=END_DATE,
        top_k=TOP_K, rebalance_days=REBALANCE_DAYS,
        position_pct=POSITION_PCT, initial_capital=INITIAL_CAPITAL,
        trade_mode=mode, verbose=False, log=False,
    )
    print(f'[B] {mode:5s}: return={result.strategy_return:.2f}%  dd={result.max_drawdown:.2f}%  hs300={result.hs300_return:.2f}%  win={result.rebalance_stats['win_rate']:.1f}%  avg={result.rebalance_stats['avg_return']:+.2f}%  ({time.time()-t0:.2f}s)')

# 验证一致性
print('\n结论: 方法A(加载模型) 与 方法B(预测缓存) 结果完全一致 ✓')

[B] close: return=27.31%  dd=3.04%  hs300=9.53%  win=100.0%  avg=+3.87%  (0.11s)
[B] open : return=4.97%  dd=6.89%  hs300=9.53%  win=0.0%  avg=-1.35%  (0.10s)

结论: 方法A(加载模型) 与 方法B(预测缓存) 结果完全一致 ✓


## 遍历所有实验，一次性缓存 + 回测

In [7]:
import glob
from tqdm import tqdm

BASE_DIR = "../model"
MODEL_TYPES = [
                "bayes_itransformer_74_3",
                "search_itransformer_74_3",
                "bayes_dlinear_74_3",
                "bayes_lstm_74_3",
                "bayes_gru_74_3",
                "search_tcn_74_3",
               ]

EXPERIMENTS = []
for mt in MODEL_TYPES:
    for exp_dir in sorted(glob.glob(f"{BASE_DIR}/{mt}/exp_*")):
        if os.path.exists(f"{exp_dir}/best_model_sliding.pth"):
            EXPERIMENTS.append((exp_dir, "best_model_sliding.pth"))
        if os.path.exists(f"{exp_dir}/best_model.pth"):
            EXPERIMENTS.append((exp_dir, "best_model.pth"))
print(f"共 {len(EXPERIMENTS)} 个实验")

all_results = []
for exp_dir, mf in tqdm(EXPERIMENTS, desc="回测"):
    cache_key = f"{exp_dir}/{mf}"
    safe_name = cache_key.replace("\\", "/").replace('../', '').replace('./', '').replace('/', '_')
    cache_path = os.path.join(CACHE_DIR, f"{safe_name}.pkl")
    
    if not os.path.exists(cache_path):
        try:
            bt = ETFBacktester.from_cached_data(
                model_dir=exp_dir, cached_data=cached_data,
                cached_features=cached_features, device='cpu',
                model_file=mf, verbose=False,
            )
            preds = bt.generate_predictions_dict(start_date=START_DATE, end_date=END_DATE, rebalance_days=REBALANCE_DAYS, first_rebalance_date=START_DATE)
            with open(cache_path, 'wb') as f:
                pickle.dump(preds, f)
            del bt.model, bt
        except Exception as e:
            print(f'FAIL gen {cache_key}: {e}')
            continue
    else:
        with open(cache_path, 'rb') as f:
            preds = pickle.load(f)
    
    for mode in ['close', 'open']:
        try:
            r = run_backtest_from_predictions(
                predictions_dict=preds, data_path=DATA_PATH,
                start_date=START_DATE, end_date=END_DATE,
                top_k=TOP_K, rebalance_days=REBALANCE_DAYS,
                position_pct=POSITION_PCT, initial_capital=INITIAL_CAPITAL,
                trade_mode=mode, verbose=False, log=False,
            )
            all_results.append({
                'experiment': exp_dir.replace("\\", "/").split('/')[-2] + '/' + exp_dir.replace("\\", "/").split('/')[-1],
                'model_file': mf, 'trade_mode': mode,
                'return': r.strategy_return,
                'dd': r.max_drawdown,
                'hs300': r.hs300_return,
                'excess': r.excess_return,
                'win_rate': r.rebalance_stats.get('win_rate', 0),
                'avg_return': r.rebalance_stats.get('avg_return', 0),
                'rebalances': r.rebalance_stats.get('total', 0),
            })
        except Exception as e:
            print(f'FAIL backtest {cache_key} {mode}: {e}')

df = pd.DataFrame(all_results)
for mode in ['close', 'open']:
    sub = df[df['trade_mode'] == mode].sort_values('return', ascending=False).head(5)
    print(f'\n=== {mode} Top 5 ===')
    for _, r in sub.iterrows():
        print(f'  {r["experiment"]:35s} {r["model_file"]:25s} return={r["return"]:6.2f}%  dd={r["dd"]:5.2f}%  win={r["win_rate"]:5.1f}%  avg={r["avg_return"]:+5.2f}%')

共 770 个实验


回测: 100%|██████████| 770/770 [02:34<00:00,  5.00it/s]


=== close Top 5 ===
  search_itransformer_74_3/exp_25     best_model_sliding.pth    return= 27.33%  dd= 4.03%
  search_itransformer_74_3/exp_25     best_model.pth            return= 27.33%  dd= 4.03%
  search_itransformer_74_3/exp_54     best_model_sliding.pth    return= 27.31%  dd= 3.04%
  search_itransformer_74_3/exp_54     best_model.pth            return= 27.31%  dd= 3.04%
  bayes_dlinear_74_3/exp_2            best_model_sliding.pth    return= 25.57%  dd= 3.42%

=== open Top 5 ===
  search_itransformer_74_3/exp_37     best_model.pth            return= 22.20%  dd= 2.39%
  search_itransformer_74_3/exp_4      best_model_sliding.pth    return= 19.43%  dd= 2.44%
  search_itransformer_74_3/exp_2      best_model_sliding.pth    return= 19.21%  dd= 6.28%
  search_itransformer_74_3/exp_28     best_model.pth            return= 18.86%  dd= 3.22%
  search_itransformer_74_3/exp_55     best_model_sliding.pth    return= 18.48%  dd= 7.05%


In [8]:
df = pd.DataFrame(all_results)
for mode in ['close', 'open']:
    sub = df[df['trade_mode'] == mode].sort_values('return', ascending=False).head(10)
    display(sub[['experiment', 'model_file', 'return', 'dd', 'hs300', 'excess']])

,experiment,model_file,return,dd,hs300,excess
280,search_itransformer_74_3/exp_25,best_model_sliding.pth,27.33,4.03,9.53,17.80
282,search_itransformer_74_3/exp_25,best_model.pth,27.33,4.03,9.53,17.80
408,search_itransformer_74_3/exp_54,best_model_sliding.pth,27.31,3.04,9.53,17.79
410,search_itransformer_74_3/exp_54,best_model.pth,27.31,3.04,9.53,17.79
544,bayes_dlinear_74_3/exp_2,best_model_sliding.pth,25.57,3.42,9.53,16.04
8,bayes_itransformer_74_3/exp_10,best_model_sliding.pth,25.50,4.43,9.53,15.97
10,bayes_itransformer_74_3/exp_10,best_model.pth,25.50,4.43,9.53,15.97
502,bayes_dlinear_74_3/exp_1,best_model.pth,25.34,1.48,9.53,15.81
500,bayes_dlinear_74_3/exp_1,best_model_sliding.pth,25.34,1.48,9.53,15.81
334,search_itransformer_74_3/exp_37,best_model.pth,25.29,2.36,9.53,15.76


,experiment,model_file,return,dd,hs300,excess
335,search_itransformer_74_3/exp_37,best_model.pth,22.20,2.39,9.53,12.68
345,search_itransformer_74_3/exp_4,best_model_sliding.pth,19.43,2.44,9.53,9.91
257,search_itransformer_74_3/exp_2,best_model_sliding.pth,19.21,6.28,9.53,9.68
295,search_itransformer_74_3/exp_28,best_model.pth,18.86,3.22,9.53,9.33
413,search_itransformer_74_3/exp_55,best_model_sliding.pth,18.48,7.05,9.53,8.95
443,search_itransformer_74_3/exp_61,best_model.pth,18.15,5.32,9.53,8.62
287,search_itransformer_74_3/exp_26,best_model.pth,18.13,5.06,9.53,8.60
363,search_itransformer_74_3/exp_43,best_model.pth,18.03,4.41,9.53,8.50
403,search_itransformer_74_3/exp_52,best_model.pth,17.65,7.10,9.53,8.12
401,search_itransformer_74_3/exp_52,best_model_sliding.pth,17.65,7.10,9.53,8.12
